In [1]:
!pip install torch==2.2.0 torchtext==0.16.2 portalocker

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.data.dataset import random_split
import re
from collections import Counter, OrderedDict

print(f"PyTorch verzija: {torch.__version__}")
print(f"CUDA dostupan: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Uređaj koji se koristi: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.4/755.4 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start
    self.io_loop.start()
  File "/usr/local/lib/python3.12/dist-package

PyTorch verzija: 2.2.0+cu121
CUDA dostupan: False
Uređaj koji se koristi: cpu


In [2]:
import kagglehub
import pandas as pd
from sklearn.model_selection import train_test_split


path = kagglehub.dataset_download("rosemeenshaikh/top-10000-tmdbs-highest-rated-movies")
print(f"Dataset path: {path}")


import os
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))

print(df.shape)
df.head()

Using Colab cache for faster access to the 'top-10000-tmdbs-highest-rated-movies' dataset.
Dataset path: /kaggle/input/top-10000-tmdbs-highest-rated-movies
(10000, 10)


,adult,id,original_language,original_title,overview,popularity,release_date,title,vote_average,vote_count
0,False,278,en,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,46.3708,1994-09-23,The Shawshank Redemption,8.718,30171
1,False,238,en,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...",42.0006,1972-03-14,The Godfather,8.686,22787
2,False,240,en,The Godfather Part II,In the continuing saga of the Corleone crime f...,26.8671,1974-12-20,The Godfather Part II,8.571,13812
3,False,424,en,Schindler's List,The true story of how businessman Oskar Schind...,24.2944,1993-12-15,Schindler's List,8.567,17341
4,False,389,en,12 Angry Men,The defense and the prosecution have rested an...,19.4971,1957-04-10,12 Angry Men,8.559,9908


In [3]:
df = df.dropna(subset=['overview']).reset_index(drop=True)

train_df, test_df = train_test_split(df, test_size=0.15, random_state=1)
train_df, valid_df = train_test_split(train_df, test_size=0.2, random_state=1)

print(f"Trening primeraka: {len(train_df)}")
print(f"Validacionih primeraka: {len(valid_df)}")
print(f"Test primeraka: {len(test_df)}")

Trening primeraka: 6798
Validacionih primeraka: 1700
Test primeraka: 1500


In [4]:
def tokenizer(text):
    text = text.lower()
    text = BeautifulSoup(text, "html.parser").get_text()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\W+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.split()